# Part 1 — Three Kinds of Problems (and why we'll need Agent-Based Modeling)

You don't need to read the code closely in this notebook. Just run each cell, look at the picture it makes, and read the short explanation before and after it. The goal is to walk away understanding **three different kinds of problems science deals with** — because that's what motivates everything else in this course.

**The story, in one paragraph:** In 1948, a scientist named Warren Weaver noticed that science had gotten really good at two very different kinds of problems, but was still mostly stuck on a third kind. We're going to look at one example of each. The third kind is the reason Agent-Based Modeling (ABM) exists at all — so pay closest attention to that one.


In [ ]:
# We just need three toolkits:
# - numpy: for doing math on lists of numbers quickly
# - matplotlib: for drawing pictures/graphs
# - scipy: has a ready-made tool for solving certain equations
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True

print("Ready to go.")


---
## Case 1 — "Simple" problems

**Everyday example:** you throw a ball. If you know two things — how fast you threw it, and at what angle — you can calculate *exactly* where it will land. No guessing, no surprises. Every time you repeat it with the same angle and speed, you get the same answer.

Weaver called this a **problem of simplicity**: only a couple of variables, and they fully determine the outcome. This is the kind of problem that 17th–19th century physics solved brilliantly.

Let's just watch a ball fly at a few different angles and confirm: yes, this is fully predictable.


In [ ]:
# This function calculates where a ball goes over time, given how it was thrown.
# You don't need to understand the physics formula -- just know that
# "angle in, full flight path out" is ALL this needs.

g = 9.81   # gravity (how fast things fall)
v0 = 20.0  # launch speed

def ball_path(angle_degrees):
    angle = np.radians(angle_degrees)
    vx, vy = v0 * np.cos(angle), v0 * np.sin(angle)
    flight_time = 2 * vy / g
    t = np.linspace(0, flight_time, 200)
    x = vx * t
    y = vy * t - 0.5 * g * t**2
    return x, y


In [ ]:
# Now let's actually throw the ball at a few different angles and watch what happens.

plt.figure()
for angle in [20, 35, 45, 60, 75]:
    x, y = ball_path(angle)
    plt.plot(x, y, label=f"{angle}°")

plt.title("A ball thrown at different angles\n(same 2 numbers -> exact answer, every single time)")
plt.xlabel("distance traveled")
plt.ylabel("height")
plt.legend(title="launch angle")
plt.ylim(bottom=0)
plt.show()


**What to notice:** every curve is smooth and exact. Nothing here is left to chance. Give the computer the same 2 numbers (angle, speed) and it will draw the exact same curve, forever.

**Takeaway for Case 1:** few variables + a known formula = perfectly predictable. Great for balls, planets, pendulums. Not so useful once things get messy — which is next.


---
## Case 2 — "Disorganized" problems

**Everyday example:** picture a crowded plaza where hundreds of people are wandering around with no particular destination — bumping into each other, changing direction randomly. Nobody's path is predictable. But if you ask "roughly how fast is the *average* person moving?", that answer turns out to be very stable and boring to predict, even though every individual is doing something different.

This is the same idea behind gas molecules bouncing around in the air, or a huge crowd, or a massive telephone network. Weaver called this **disorganized complexity**: tons of individual parts, each behaving unpredictably on its own — but the *group average* is remarkably steady.

Let's simulate hundreds of little particles bouncing around a box and see this for ourselves.


In [ ]:
# Step 1: create the particles.
# Each particle gets a random starting position and a random starting speed/direction.

rng = np.random.default_rng(7)

n_particles = 800
box_size = 10.0

positions = rng.uniform(0, box_size, size=(n_particles, 2))
velocities = rng.normal(0, 1.0, size=(n_particles, 2))

print(f"Created {n_particles} particles bouncing around in a box.")


In [ ]:
# Step 2: let time pass. Each little step, every particle moves a tiny bit,
# and bounces off the walls if it hits one (that's the "if hit_low or hit_high" part).
# We'll also remember where 5 sample particles went, just so we can look at
# a few individual paths afterward.

n_steps = 300
sample_trajectories = np.zeros((n_steps, 5, 2))

for step in range(n_steps):
    positions += velocities * 0.05
    for dim in range(2):
        hit_low = positions[:, dim] < 0
        hit_high = positions[:, dim] > box_size
        velocities[hit_low | hit_high, dim] *= -1
        positions[:, dim] = np.clip(positions[:, dim], 0, box_size)
    sample_trajectories[step] = positions[:5]

print("Simulation finished.")


In [ ]:
# Step 3a: look at just a FEW individual particles' paths.
plt.figure()
for i in range(5):
    plt.plot(sample_trajectories[:, i, 0], sample_trajectories[:, i, 1], alpha=0.8)
plt.title("A few individual particle paths\n(messy, different every time you run it)")
plt.xlim(0, box_size)
plt.ylim(0, box_size)
plt.show()


In [ ]:
# Step 3b: now zoom OUT and look at the group as a whole -- how fast is everybody moving?
speeds = np.linalg.norm(velocities, axis=1)

plt.figure()
plt.hist(speeds, bins=30, color="steelblue", edgecolor="white")
plt.title(f"Speed of all {n_particles} particles, all at once\n(smooth & predictable, despite the mess above)")
plt.xlabel("speed")
plt.show()

print(f"Average speed: {speeds.mean():.3f}")
print("Try re-running this whole notebook with a different seed number --")
print("individual paths will look totally different, but this average will barely move.")


**What to notice:** the single-particle paths (Step 3a) are genuinely unpredictable — you couldn't guess where any one of them will be next. But the group-level picture (Step 3b) is smooth, stable, and would look almost identical if we reran this with different random numbers.

**Takeaway for Case 2:** you can't predict individuals, but you *can* predict the group average, almost exactly. This is what statistics is built for.


---
## Case 3 — "Organized" problems (and where the old tools start to struggle)

**Everyday example:** an ecosystem. Grass grows. Rabbits eat grass. Foxes eat rabbits. Parasites live on foxes. These aren't independent and they're not random — they're all **tangled together**. More rabbits means more food for foxes, which means more foxes, which means fewer rabbits, which eventually means fewer foxes again... round and round.

Weaver called this **organized complexity**: not a huge random crowd (Case 2), and not just 2–3 variables (Case 1) — but a moderate number of variables that are all *interdependent*, forming what he called "an organic whole."

Historically, scientists tried to attack these problems the same way they'd attacked Case 1: **write one equation per variable**, describing how the whole *population* changes on average — no individuals, just totals. Let's try exactly that.


In [ ]:
# Step 1: write down how each population changes, based on the OTHER populations.
# You do NOT need to follow the math here. The important thing to notice is:
# we are describing GROUP TOTALS (all grass, all rabbits, all foxes...) with
# a handful of equations -- nobody is simulating one individual rabbit.

def ecosystem(populations, t):
    grass, rabbits, foxes, parasites = populations
    d_grass     = 1.0 * grass * (1 - grass / 50) - 0.04 * grass * rabbits
    d_rabbits   = 0.02 * grass * rabbits - 0.05 * rabbits * foxes - 0.01 * rabbits * parasites - 0.1 * rabbits
    d_foxes     = 0.02 * rabbits * foxes - 0.03 * foxes * parasites - 0.2 * foxes
    d_parasites = 0.015 * foxes * parasites + 0.005 * rabbits * parasites - 0.15 * parasites
    return [d_grass, d_rabbits, d_foxes, d_parasites]

print("Equations defined. Nothing simulated yet -- next cell actually solves them.")


In [ ]:
# Step 2: solve the equations forward in time and plot the four populations.

t = np.linspace(0, 200, 2000)
starting_populations = [40, 9, 5, 2]   # grass, rabbits, foxes, parasites
solution = odeint(ecosystem, starting_populations, t)

plt.figure()
for i, name in enumerate(["Grass", "Rabbits", "Foxes", "Parasites"]):
    plt.plot(t, solution[:, i], label=name)
plt.title("Four interdependent populations over time\n(the wobbling pattern EMERGES from how they're tangled together)")
plt.xlabel("time")
plt.ylabel("population size")
plt.legend()
plt.show()


**What to notice:** this actually works! The populations rise and fall in a coordinated, self-sustaining rhythm — an interesting pattern that comes purely from the interdependence, not from randomness.

**But here's the catch — and this is the important part for the rest of the course:**

To write those equations, we had to already *know*, in advance, exactly how grass, rabbits, foxes, and parasites affect each other. And every single rabbit was treated as identical and interchangeable — there's no individual rabbit, no location, no "this particular rabbit happened to run into that particular fox." We only ever simulated *totals*.

That's a real limitation. What if:
- we *don't* know the right group-level equations ahead of time, or
- individual differences and individual encounters actually matter (who's where, who meets whom, plain luck)?

This is exactly the gap **Agent-Based Modeling (ABM)** was built to fill: instead of writing one equation for the whole group, you give a simple rule to *each individual*, let them interact, and watch whatever pattern **emerges** on its own — nobody tells the model in advance what the big picture should look like.

That's what the next notebook does.


## Quick recap

| | Case 1: Simple | Case 2: Disorganized | Case 3: Organized |
|---|---|---|---|
| everyday example | throwing a ball | a random crowd / gas | an ecosystem |
| how many things involved | 2–3 | thousands, independent | a handful, all tangled together |
| can you predict one individual? | yes, exactly | no | it depends on all the others |
| can you predict the group? | it *is* the individual | yes, very precisely | only if you already know the right equations |
| how we solved it here | one formula | statistics / averages | equations for the whole population (no individuals) |

**Next up:** we throw away the equations entirely, build individual little "agent" walkers with simple personal rules, and see what pattern emerges — completely on its own.
